# Amazon Beauty RQ-VAE pipeline

End-to-end run on the Amazon Product Reviews (Beauty) dataset:
1. Build embeddings (sentence-t5-base, 768d)
2. Train the RQ-VAE
3. Generate Semantic IDs

All artifacts land under `outputs/amazon_beauty_*` because `name: amazon_beauty` in the config drives the paths.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

: 

In [ ]:
import os
import sys
from pathlib import Path

# Absolute path of your local rq-vae/ checkout. Hardcoded because some
# Jupyter setups run with cwd=/tmp, so detecting via Path.cwd() doesn't work.
ROOT = Path("/content/drive/MyDrive/tiger/rq-vae")
assert (ROOT / "src" / "amazon_beauty_dataloader.py").exists(), f"bad ROOT: {ROOT}"

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG = str(ROOT / "configs" / "amazon_beauty_config.yaml")

## 1. Build embeddings

In [4]:
from src.amazon_beauty_dataloader import prepare
from src.config import load_config

cfg = load_config(CONFIG)
prepare(cfg, download=True)

## 2. Train the RQ-VAE

In [5]:
from src.train import train

# Reinit dead codes every 1000 steps instead of the config default (250).
# ~253 steps/epoch here, so this fires roughly once every 4 epochs.
cfg["train"]["reinit_every"] = 1000

history = train(cfg)

In [10]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
# train and eval recon span many orders of magnitude (train starts ~1e16),
# so use a log y-axis on each so the later, smaller values stay readable
axes[0].plot(epochs, [h["train_recon"] for h in history], color="tab:blue")
axes[0].set_yscale("log")
axes[0].set_title("train recon loss (log)"); axes[0].set_xlabel("epoch")
axes[1].plot(epochs, [h["recon_loss"] for h in history], color="tab:orange")
axes[1].set_yscale("log")
axes[1].set_title("eval recon loss (log)"); axes[1].set_xlabel("epoch")
for l in range(len(history[0]["utilization"])):
    axes[2].plot(epochs, [h["utilization"][l] for h in history], label=f"L{l}")
axes[2].set_title("codebook utilization"); axes[2].set_xlabel("epoch"); axes[2].legend()
axes[3].plot(epochs, [h["sid_unique_fraction"] for h in history])
axes[3].set_title("unique SID fraction"); axes[3].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 3. Generate Semantic IDs

In [7]:
from src.generate_sids import generate

ckpt = Path(cfg["output"]["checkpoints_dir"]) / "best.pt"
generate(cfg, str(ckpt))

In [8]:
import json
import pandas as pd

sids = pd.read_csv(cfg["output"]["sids_csv"])
print(f"{len(sids)} items")
display(sids.head(10))

with open(cfg["output"]["metrics_json"]) as f:
    print(json.dumps(json.load(f), indent=2))